# 02 — Features comportementales (op_03)

On part de la baseline du notebook 01 (features row-level + solde + fréquences fold-safe)
et on mesure en A/B le gain des **features comportementales** (couple émetteur-destinataire,
activité par compte, degrés bipartites), toutes apprises sur le **passé** (anti-fuite).

Baseline de référence (notebook 01) : AP temporelle **0.3669**, fold 4 **0.3483**.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything
from src.features.temporal import balance_features
from src.features.behavioral import behavioral_features
seed_everything(42)
DATA = ROOT / "data"

In [ ]:
train = pd.read_csv(DATA / "train.csv")
tr03 = train[op03_mask(train)].reset_index(drop=True)
y = tr03[C.TARGET].values
folds = list(time_folds(tr03[C.PERIOD]))
print("op_03 train:", tr03.shape, "| taux fraude:", round(y.mean(), 4))

In [ ]:
EPS = 1e-6

def base_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def make_model():
    try:
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                                  learning_rate=0.05, iterations=600, random_seed=42, verbose=False)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=42)

# features row-level calculées une fois (ne dépendent pas du fold)
Xbase = base_features(tr03)
print("features de base:", Xbase.shape[1])

## CV temporelle, avec / sans features comportementales
Les agrégations comportementales sont recalculées **par fold sur le passé** (`ref = train du fold`).

In [ ]:
def run_cv(use_behavioral):
    oof = np.zeros(len(y))
    last_model, last_cols = None, None
    for tr_idx, va_idx in folds:
        ref = tr03.iloc[tr_idx]
        Xtr = add_freq(Xbase.iloc[tr_idx].reset_index(drop=True), ref, ref)
        Xva = add_freq(Xbase.iloc[va_idx].reset_index(drop=True), tr03.iloc[va_idx], ref)
        if use_behavioral:
            btr = behavioral_features(ref, ref).reset_index(drop=True)
            bva = behavioral_features(tr03.iloc[va_idx], ref).reset_index(drop=True)
            Xtr = pd.concat([Xtr, btr], axis=1)
            Xva = pd.concat([Xva, bva], axis=1)
        m = make_model()
        m.fit(Xtr, y[tr_idx])
        oof[va_idx] = m.predict_proba(Xva)[:, 1]
        last_model, last_cols = m, Xtr.columns
    return oof, last_model, last_cols

oof_base, _, _ = run_cv(use_behavioral=False)
oof_beh, model_beh, cols_beh = run_cv(use_behavioral=True)

ap_base = evaluate_ap(y, oof_base)
ap_beh = evaluate_ap(y, oof_beh)
print(f"AP SANS comportemental : {ap_base:.4f}")
print(f"AP AVEC comportemental : {ap_beh:.4f}")
print(f"Gain global             : {ap_beh - ap_base:+.4f}")
for k, (_, va) in enumerate(folds):
    print(f"  fold {k}: {evaluate_ap(y[va], oof_base[va]):.4f} -> {evaluate_ap(y[va], oof_beh[va]):.4f}")

## Quelles features comportementales portent le signal ?

In [ ]:
if hasattr(model_beh, "get_feature_importance"):
    imp = model_beh.get_feature_importance()
else:
    imp = getattr(model_beh, "feature_importances_", np.zeros(len(cols_beh)))
fi = pd.Series(imp, index=cols_beh).sort_values(ascending=False)
print("Top 12 features (dernier fold) :")
print(fi.head(12).round(2))

## Décision
- **Gain fold 4 > +0.01** → garder, committer, et continuer le comportemental (dynamique récente).
- **Gain marginal** → identifier dans l'importance ci-dessus quelles features valent le coup,
  jeter le reste, et passer à la dynamique temporelle (intervalle entre tx, comptage récent).